#  **Data Collection and Preprocessing**

### Import Libraries

In [1]:
import sys
import os

# Add parent directory to path so we can import from src
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, parent_dir)

In [2]:
from src.preprocessing import (
    clean_review_text,
    display_app_info,
    review_dataframe,
    remove_duplicates,
    handle_missing_data,
    normalize_dates,
    validate_rating,
    preprocessing_report,
    save_cleaned_data,
    count_review_languages,
    remove_non_english_reviews
)
from src.data_scrapping import scrap_reviews

### Web Scraping

#### App metadata

In [3]:
CBE_APP_ID = 'com.combanketh.mobilebanking'
display_app_info(CBE_APP_ID)

Commercial Bank of Ethiopia App Info
App Title   : Commercial Bank of Ethiopia
Current Score: 4.289121
Total Ratings: 48,358
Total Reviews: 9,310
Installs     : 5,000,000+


#### Scrape reviews

In [4]:
reviews = scrap_reviews(app_id=CBE_APP_ID, num_reviews=1000)

Scraping reviews for com.combanketh.mobilebanking...
Collected 1000 raw reviews


#### Collect review text, rating, review date, bank , source

In [5]:
# Let's inspect what a single raw review looks like
print("Keys in a single review:")
print(list(reviews[0].keys()))

print("\nFirst raw review (sample):")
for key, value in reviews[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: bfc27568-1471-4937-934f-e5325ea96f46
  userName: Alem Melese
  userImage: https://play-lh.googleusercontent.com/a-/ALV-UjVDumA051FtjvkehOz-itfdsb6wl_VMEZTafK-CMkEVmTd17IOH
  content: nice app
  score: 5
  thumbsUpCount: 0
  reviewCreatedVersion: 5.3.0
  at: 2026-05-14 18:28:23
  replyContent: None
  repliedAt: None
  appVersion: 5.3.0


In [6]:
df = review_dataframe(reviews, app_info={'title': 'CBE Bank'})

print(f"Shape: {df.shape}")
df.head()

Shape: (1000, 6)


,review_id,review,rating,date,bank,source
0,bfc27568-1471-4937-934f-e5325ea96f46,nice app,5,2026-05-14 18:28:23,CBE Bank,Google Play
1,9d8ba9a5-4899-4af9-bef1-f12a5dfd7f3a,formative,5,2026-05-14 16:49:46,CBE Bank,Google Play
2,07dea887-2fcf-446c-ac8d-753be7c90256,best app for financial activities 🙌,5,2026-05-14 16:46:29,CBE Bank,Google Play
3,74bf72d1-d6ce-4bd7-9a4d-3950846e5aba,yoroo namaste 🙏 ♥️ ❤️ 💖 💖,5,2026-05-14 12:13:13,CBE Bank,Google Play
4,0e691771-0024-4326-b71f-1685b91dc83d,incredible,5,2026-05-14 10:53:56,CBE Bank,Google Play


## Preprocessing

#### Remove duplicate reviews

In [7]:
df_clean = df.copy()

In [8]:
df_clean = remove_duplicates(df_clean)

Removed 0 duplicate reviews
Remaining: 1000 reviews


#### Handle missing values

In [9]:
df_clean =handle_missing_data(df_clean)

Removed 0 rows with missing critical data
Remaining: 1000 reviews


#### Normalize dates to YYYY-MM-DD format

In [10]:
df_clean = normalize_dates(df_clean)

Before normalization:
0   2026-05-14 18:28:23
1   2026-05-14 16:49:46
2   2026-05-14 16:46:29
dtype: datetime64[us]

After normalization:
0    2026-05-14
1    2026-05-14
2    2026-05-14
dtype: str

Date range: 2025-12-13 to 2026-05-14


#### Handle incorrect ratings

In [11]:
df_clean = validate_rating(df_clean)

All ratings are valid (1-5).
Remaining: 1000 reviews


#### Clean review text

In [12]:
df_clean['review'] = df_clean['review'].apply(clean_review_text)

print("Sample cleaned reviews:")
print(df_clean['review'].head(10).to_string())

Sample cleaned reviews:
0                              nice app
1                             formative
2    best app for financial activities 
3                    yoroo namaste     
4                            incredible
5         best app for financial sector
6               it's a good application
7                         thank you cbe
8                               is good
9                                   wow


#### Save the cleaned dataset

In [13]:
# Select only the 5 required columns in the right order
df_clean = df_clean[['review', 'rating', 'date', 'bank', 'source']]

# Sort by date (newest first) for clean presentation
df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean.shape}")
df_clean.head(10)

Final dataset shape: (1000, 5)


,review,rating,date,bank,source
0,nice app,5,2026-05-14,CBE Bank,Google Play
1,best app for financial activities,5,2026-05-14,CBE Bank,Google Play
2,yoroo namaste,5,2026-05-14,CBE Bank,Google Play
3,incredible,5,2026-05-14,CBE Bank,Google Play
4,best app for financial sector,5,2026-05-14,CBE Bank,Google Play
5,formative,5,2026-05-14,CBE Bank,Google Play
6,it's a good application,5,2026-05-13,CBE Bank,Google Play
7,thank you cbe,5,2026-05-13,CBE Bank,Google Play
8,is good,5,2026-05-13,CBE Bank,Google Play
9,wow,5,2026-05-13,CBE Bank,Google Play


In [14]:
save_cleaned_data(df_clean, output_path="../data/processed/cbe_reviews_cleaned.csv")

Cleaned data saved to ../data/processed/cbe_reviews_cleaned.csv
Saved to: ../data/processed/cbe_reviews_cleaned.csv


### Report

In [15]:
preprocessing_report(df, df_clean)

  PREPROCESSING REPORT — Awash Bank Reviews

  Raw reviews collected  :   1000
  Reviews after cleaning :   1000
  Reviews removed        :      0
  Data retention rate    : 100.0%
  Data quality           : EXCELLENT

  Date range : 2025-12-13  to  2026-05-14
Rating distribution:
  5 stars:  666  █████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  4 stars:   78  ███████████████
  3 stars:   64  ████████████
  2 stars:   44  ████████
  1 stars:  148  █████████████████████████████

  Text length stats:
    Min    : 0 characters
    Median : 13 characters
    Max    : 500 characters

  Columns in final CSV:
    - review
    - rating
    - date
    - bank
    - source

